In [8]:
%pip install nba_api pandas 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\lukin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
from nba_api.stats.static import teams 
import pandas as pd 

nba_teams_list = teams.get_teams()
df_teams = pd.DataFrame(nba_teams_list)

df_teams.head(10)

,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970
3,1610612740,New Orleans Pelicans,NOP,Pelicans,New Orleans,Louisiana,2002
4,1610612741,Chicago Bulls,CHI,Bulls,Chicago,Illinois,1966
5,1610612742,Dallas Mavericks,DAL,Mavericks,Dallas,Texas,1980
6,1610612743,Denver Nuggets,DEN,Nuggets,Denver,Colorado,1976
7,1610612744,Golden State Warriors,GSW,Warriors,San Francisco,California,1946
8,1610612745,Houston Rockets,HOU,Rockets,Houston,Texas,1967
9,1610612746,Los Angeles Clippers,LAC,Clippers,Los Angeles,California,1970


In [10]:
from nba_api.stats.endpoints import leaguegamefinder
import time 

knicks_info = df_teams[df_teams['full_name'] == 'New York Knicks']
knicks_id = knicks_info['id'].values[0]
print(f"O ID oficial do New York Knicks é> {knicks_id}") 

time.sleep(1)

game_finder = leaguegamefinder.LeagueGameFinder(team_id_nullable=knicks_id)
df_knicks_game = game_finder.get_data_frames()[0]
df_knicks_game.head(5) 

O ID oficial do New York Knicks é> 1610612752


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,42025,1610612752,NYK,New York Knicks,0042500405,2026-06-13,NYK @ SAS,W,240,94,...,0.714,13,35,48,14,8,4,10,21,4.0
1,42025,1610612752,NYK,New York Knicks,0042500404,2026-06-10,NYK vs. SAS,W,240,107,...,0.714,8,31,39,23,6,4,13,21,1.0
2,42025,1610612752,NYK,New York Knicks,0042500403,2026-06-08,NYK vs. SAS,L,239,111,...,0.818,12,34,46,18,4,6,13,23,-4.0
3,42025,1610612752,NYK,New York Knicks,0042500402,2026-06-05,NYK @ SAS,W,240,105,...,0.762,10,34,44,29,11,6,12,23,1.0
4,42025,1610612752,NYK,New York Knicks,0042500401,2026-06-03,NYK @ SAS,W,239,105,...,0.889,10,39,49,20,8,4,8,23,10.0


In [11]:
colunas_importantes = ['GAME_DATE', 'MATCHUP', 'WL', 'PTS', 'PLUS_MINUS']
df_knicks = df_knicks_game[colunas_importantes].copy()

df_knicks['GAME_DATE'] = pd.to_datetime(df_knicks['GAME_DATE'])
df_knicks['LOCAL'] = df_knicks['MATCHUP'].apply(lambda x: 'Fora' if '@' in x else 'Casa')
df_knicks = df_knicks.dropna(subset=['WL'])

analise_casa_fora = pd.crosstab(df_knicks['LOCAL'], df_knicks['WL'], margins=True)
analise_casa_fora['WIN_RATE_%'] = round((analise_casa_fora['W'] / analise_casa_fora['All']) * 100, 2) 
analise_casa_fora 

WL,L,W,All,WIN_RATE_%
LOCAL,,,,
Casa,797,1115,1912,58.32
Fora,1174,747,1921,38.89
All,1971,1862,3833,48.58


In [12]:
%pip install matplotlib seaborn plotly 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\lukin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


DESEMPENHO DO NEW YORK KNICKS (DENTRO E FORA DE CASA)


In [13]:
import plotly.graph_objects as go

vitorias_casa = analise_casa_fora.loc['Casa', 'W']
derrotas_casa = analise_casa_fora.loc['Casa', 'L']
win_rate_casa = analise_casa_fora.loc['Casa', 'WIN_RATE_%']

vitorias_fora = analise_casa_fora.loc['Fora', 'W']
derrotas_fora = analise_casa_fora.loc['Fora', 'L']
win_rate_fora = analise_casa_fora.loc['Fora', 'WIN_RATE_%']

fig = go.Figure()

CORES_KNICKS = {'W': '#F58426', 'L': '#006BB6'}

fig.add_trace(go.Bar(
    x=['Casa', 'Fora'],
    y=[vitorias_casa, vitorias_fora],
    name='Vitórias (W)',
    marker_color=[CORES_KNICKS['W'], CORES_KNICKS['W']],
    text=[f"{vitorias_casa}<br><b>({win_rate_casa}%)</b>", f"{vitorias_fora}<br><b>({win_rate_fora}%)</b>"],
    textposition='auto',
    textfont=dict(size=14, color='white')
))

fig.add_trace(go.Bar(
    x=['Casa', 'Fora'],
    y=[derrotas_casa, derrotas_fora],
    name='Derrotas (L)',
    marker_color=[CORES_KNICKS['L'], CORES_KNICKS['L']],
    text=[f"{derrotas_casa}", f"{derrotas_fora}"],
    textposition='auto',
    textfont=dict(size=14, color='white')
))

fig.update_layout(
    title={
        'text': "<b>Desempenho New York Knicks</b><br><sup>Comparativo de Vitórias vs Derrotas por Local do Jogo</sup>",
        'y': 0.93,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=20)
    },
    barmode='group',
    bargap=0.25,
    bargroupgap=0.1,
    template='plotly_white',
    paper_bgcolor='#F9F9F9',
    plot_bgcolor='#FFFFFF',
    font=dict(family="Arial, sans-serif", size=12, color="#333333"),
    legend=dict(
        title="Resultado",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
xaxis=dict(title="<b>Local da Partida</b>", tickfont=dict(size=14)), 
yaxis=dict(title="<b>Quantidade de Jogos</b>", showgrid=True, gridcolor='#E5E5E5'),
margin=dict(l=50, r=50, t=100, b=50),
height=450
)
fig.show()

In [14]:
from sqlalchemy import create_engine
import pandas as pd 

USUARIO = 'postgres'
SENHA = 'Postgres1312'
HOST = 'localhost'
PORTA = '5432'
BANCO_DE_DADOS = 'nba_db'

string_conexao = f'postgresql://{USUARIO}:{SENHA}@{HOST}:{PORTA}/{BANCO_DE_DADOS}?client_encoding=utf-8'
engine = create_engine(string_conexao)

try:
    df_knicks.to_sql(
        name='tb_knicks_jogos',
        con=engine,
        if_exists='replace',
        index=False
    )
    print("A tabela 'tb_knicks_jogos' foi grava nos PostgreSQL!!!")
except Exception as e:
    print("Mensagem de erro:")
    print(e)



A tabela 'tb_knicks_jogos' foi grava nos PostgreSQL!!!


In [15]:

from nba_api.stats.endpoints import commonteamroster, playergamelog
import time
import pandas as pd 

time.sleep(1)

roster = commonteamroster.CommonTeamRoster(team_id=knicks_id)
df_roster = roster.get_data_frames()[0]

colunas_roster = ['PLAYER_ID', 'PLAYER', 'NUM', 'POSITION', 'HEIGHT', 'WEIGHT']
df_roster = df_roster[colunas_roster]
df_roster.head(5)

,PLAYER_ID,PLAYER,NUM,POSITION,HEIGHT,WEIGHT
0,203903,Jordan Clarkson,00,G,6-5,194
1,1641794,Dillon Jones,,F,6-5,235
2,1630540,Miles McBride,2,G,6-2,195
3,1628404,Josh Hart,3,G,6-5,215
4,1642359,Pacôme Dadiet,4,F,6-9,210


Filtrando e Analisando o Desempenho de Jalen Brunson Ponto a Ponto 
                    (ARMADOR DO KNICKS)
            


In [16]:
from nba_api.stats.endpoints import playergamelog
import pandas as pd
import time

brunson_info = df_roster[df_roster['PLAYER'].str.contains('Brunson', case=False, na=False)]
brunson_id = brunson_info['PLAYER_ID'].values[0]
print(f"O ID oficial do Jalen Brunson na NBA é: {brunson_id}")

time.sleep(1)

brunson_games = playergamelog.PlayerGameLog(player_id=brunson_id)
df_brunson = brunson_games.get_data_frames()[0]

df_brunson['GAME_DATE'] = pd.to_datetime(df_brunson['GAME_DATE'])
df_brunson = df_brunson.sort_values('GAME_DATE').reset_index(drop=True)
media_pontos = round(df_brunson['PTS'].mean(), 1)

print(f"Média de pontos calculada: {media_pontos} PTS por jogo.")

O ID oficial do Jalen Brunson na NBA é: 1628973
Média de pontos calculada: 26.0 PTS por jogo.


Evolução de Pontos do Jalen Brunson 


In [17]:
fig_brunson = go.Figure()

fig_brunson.add_trace(go.Scatter(
    x=df_brunson['GAME_DATE'],
    y=df_brunson['PTS'],
    mode='lines+markers',
    name='Pontos no Jogo',
    line=dict(color='#F58426', width=2),
    marker=dict(size=6, color='#F58426'),
    hovertemplate='<b>Data:</b> %{x|%d/%m/%Y}<br><b>Pontos:</b> %{y}<extra></extra>'
))

fig_brunson.add_trace(go.Scatter(
    x=df_brunson['GAME_DATE'],
    y=[media_pontos] * len(df_brunson),
    mode='lines',
    name=f'Média ({media_pontos} PTS)',
    line=dict(color='#006BB6', width=2, dash='dash')
))

fig_brunson.update_layout(
    title={
        'text': f"<b>Evolução de Pontuação: Jalen Brunson</b><br><sup>Média da Temporada: {media_pontos} pontos por jogo</sup>",
        'y': 0.93,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=18)
    },
    template='plotly_white',
    paper_bgcolor='#F9F9F9',
    plot_bgcolor='#FFFFFF',
    font=dict(family="Arial, sans-serif", size=12, color="#333333"),
    xaxis=dict(title="<b>Data da Partida</b>", showgrid=True, gridcolor='#E5E5E5'),
    yaxis=dict(title="<b>Pontos Marcados (PTS)</b>", showgrid=True, gridcolor='#E5E5E5'),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(l=50, r=50, t=100, b=50),
    height=450
)
fig_brunson.show()


In [18]:
try: 
    df_brunson.to_sql(
        name='tb_brunson_jogos',
        con=engine,
        if_exists='replace',
        index=False
    )
    print("Perfeito! A tabela 'tb_brunson_jogos' foi salva no PostgreSQL!!!")
except Exception as e:
    print("Ocorreu um erro:")
    print(e) 

Perfeito! A tabela 'tb_brunson_jogos' foi salva no PostgreSQL!!!
